In [2]:
!wget -q --show-progress -O all-the-news-3.zip "https://www.dropbox.com/scl/fi/wruzj2bwyg743d0jzd7ku/all-the-news-3.zip?rlkey=rgwtwpeznbdadpv3f01sznwxa&dl=1"
!unzip all-the-news-3.zip

all-the-news-3.zip  100%[===================>] 166.04M  80.4MB/s    in 2.1s    
Archive:  all-the-news-3.zip
  inflating: all-the-news-3.csv      


In [3]:
!pip install pandas tiktoken openai pinecone tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 18.2 MB/s eta 0:00:00


In [4]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import time
import json
from getpass import getpass
from typing import List, Dict, Any
import pandas as pd
import tiktoken
from tqdm.auto import tqdm
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
# ============================================================
# 2. CONFIGURATION
# ============================================================
INDEX_NAME = "global-news-researcher"
NAMESPACE = "__default__"
CSV_PATH = "./all-the-news-3.csv"
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIMENSION = 1536
CHAT_MODEL = "gpt-4o"
TOTAL_ROWS = 1000 # Demo size for classroom
READ_CHUNK_SIZE = 100 # Pandas reads 100 rows at a time
UPSERT_BATCH_SIZE = 50 # Safe batch size for Pinecone upsert
MAX_EMBED_TOKENS = 8000 # Token safety limit
MAX_METADATA_BYTES = 40960 # 40 KB Pinecone metadata limit
MAX_ARTICLE_CHARS = 5000 # Store only a teaching-friendly article snippet
RESET_INDEX = False # Set True only when you want to recreate the index

In [6]:
def get_secret(name: str) -> str:
    """
    Read API key from:
    1. Environment variable
    2. Google Colab userdata
    3. Manual input
    """
    value = os.environ.get(name)

    if value:
        return value

    try:
        from google.colab import userdata

        value = userdata.get(name)

        if value:
            os.environ[name] = value
            return value

    except Exception:
        pass

    value = getpass(f"Enter {name}: ")
    os.environ[name] = value

    return value

In [10]:
def list_index_names() -> List[str]:
    return [idx.name for idx in pc.list_indexes()]


# Delete the existing index if RESET_INDEX is True
if RESET_INDEX and INDEX_NAME in list_index_names():
    print(f"Deleting existing index: {INDEX_NAME}")
    pc.delete_index(INDEX_NAME)
    time.sleep(10)


# Create the index if it does not exist
if INDEX_NAME not in list_index_names():
    print(f"Creating index: {INDEX_NAME}")

    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    time.sleep(10)

else:
    print(f"Using existing index: {INDEX_NAME}")


# Connect to the index
index = pc.Index(INDEX_NAME)

Creating index: global-news-researcher


Considering the drawbacks of
metadatasize(40KB), chunk size (8192), No of
records 1000 or 2MB max upsert records. We are
going to define the utility functions

In [11]:
def safe_text(value: Any) -> str:
    """
    Convert null/NaN values into empty strings.
    """
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def truncate_by_tokens(
    text: str,
    max_tokens: int = MAX_EMBED_TOKENS
) -> str:
    """
    Keep text within the token limit.
    """
    text = safe_text(text)
    tokens = tokenizer.encode(text)

    if len(tokens) <= max_tokens:
        return text

    return tokenizer.decode(tokens[:max_tokens])


def metadata_size_bytes(metadata: Dict[str, Any]) -> int:
    """
    Calculate metadata size in bytes.
    Pinecone metadata must stay within the allowed metadata size.
    """
    return len(
        json.dumps(
            metadata,
            ensure_ascii=False
        ).encode("utf-8")
    )


def fit_metadata(metadata: Dict[str, Any]) -> Dict[str, Any]:
    """
    Trim article_content until metadata fits within the Pinecone metadata limit.
    """
    article = metadata.get("article_content", "")

    while (
        metadata_size_bytes(metadata) > MAX_METADATA_BYTES
        and len(article) > 0
    ):
        article = article[:-500]
        metadata["article_content"] = article

    if metadata_size_bytes(metadata) > MAX_METADATA_BYTES:
        metadata["article_content"] = ""

    return metadata

In [12]:
def embed_texts(texts: List[str]) -> List[List[float]]:
    """
    Convert a list of text values into embeddings.
    Each embedding will have 1536 dimensions for text-embedding-3-small.
    """
    response = openai_client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )

    return [
        item.embedding
        for item in response.data
    ]

In [13]:
def build_record(
    row_id: int,
    row: pd.Series,
    title_vector: List[float]
) -> Dict[str, Any]:
    """
    Build one Pinecone record:
    - id
    - values: title embedding
    - metadata: title, article snippet, URL, publication, date
    """
    title = safe_text(row.get("title"))

    article = safe_text(row.get("article"))
    article = truncate_by_tokens(
        article,
        MAX_EMBED_TOKENS
    )

    metadata = {
        "title": title,
        "article_content": article[:MAX_ARTICLE_CHARS],
        "url": safe_text(row.get("url")),
        "publication": safe_text(row.get("publication")),
        "date": safe_text(row.get("date")),
        "section": safe_text(row.get("section")),
        "author": safe_text(row.get("author"))
    }

    metadata = fit_metadata(metadata)

    return {
        "id": f"news-{row_id}",
        "values": title_vector,
        "metadata": metadata
    }

.INDEX NEWS ARTICLES
Stage 1 data preparation:
Read CSV in batches
Embed only titles
Store title vectors + article metadata in Pinecon

In [14]:
def index_news_articles() -> int:
    total_indexed = 0

    chunks = pd.read_csv(
        CSV_PATH,
        chunksize=READ_CHUNK_SIZE,
        nrows=TOTAL_ROWS
    )

    for chunk in tqdm(chunks, desc="Reading CSV batches"):
        prepared_rows = []

        for row_id, row in chunk.iterrows():
            title = safe_text(row.get("title"))
            article = safe_text(row.get("article"))

            if not title or not article:
                continue

            prepared_rows.append({
                "row_id": int(row_id),
                "title": title,
                "row": row
            })

        if not prepared_rows:
            continue

        # Create embeddings in a batch instead of one API call per row
        titles = [
            item["title"]
            for item in prepared_rows
        ]

        title_vectors = embed_texts(titles)

        records = []

        for item, vector in zip(prepared_rows, title_vectors):
            record = build_record(
                row_id=item["row_id"],
                row=item["row"],
                title_vector=vector
            )

            records.append(record)

        # Upsert to Pinecone in smaller batches
        for start in range(0, len(records), UPSERT_BATCH_SIZE):
            batch = records[start:start + UPSERT_BATCH_SIZE]

            index.upsert(
                vectors=batch,
                namespace=NAMESPACE
            )

            total_indexed += len(batch)

    return total_indexed


# Run the indexing process
total = index_news_articles()

print(f"Total records indexed: {total}")


Reading CSV batches: 0it [00:00, ?it/s]

Total records indexed: 998


In [16]:
stats = index.describe_index_stats()
print(stats)

DescribeIndexStatsResponse(dimension=1536, total_vector_count=998, metric='cosine', namespaces=1)


In [17]:
def search_news_titles(query: str, top_k: int = 3):
    """
    Stage 1:
    Convert the user query into an embedding.
    Search Pinecone title vectors.
    Return the top matching news records.
    """
    query_vector = embed_texts([query])[0]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
        namespace=NAMESPACE
    )

    return results

In [19]:
def print_search_results(query: str, top_k: int = 3):
 results = search_news_titles(query, top_k=top_k)
 print(f"\nQuery: {query}")
 print("=" * 80)
 for rank, match in enumerate(results.matches, start=1):
  metadata = match.metadata or {}
  print(f"\nRank: {rank}")
  print(f"Score: {match.score}")
  print(f"Title: {metadata.get('title')}")
  print(f"Publication: {metadata.get('publication')}")
  print(f"Date: {metadata.get('date')}")
  print(f"URL: {metadata.get('url')}")
 return results
print_search_results("What happened with Obama?", top_k=3)


Query: What happened with Obama?

Rank: 1
Score: 0.358225465
Title: Media obsession with a bullshit email scandal helped Trump to the White House
Publication: Vox
Date: 2016-11-09 07:36:33
URL: https://www.vox.com/policy-and-politics/2016/11/9/13570724/media-obsession-emails

Rank: 2
Score: 0.347665429
Title: In separate speeches Obama, Biden, Carter target Trump policies
Publication: Reuters
Date: 2016-06-20 00:00:00
URL: https://www.reuters.com/article/us-usa-election-biden/in-separate-speeches-obama-biden-carter-target-trump-policies-idUSKCN0Z62LO

Rank: 3
Score: 0.346805036
Title: NYPD Honcho Insulted by 'Hamilton' Star Lin-Manuel Miranda Celebrating Obama's Controversial Prisoner Release
Publication: TMZ
Date: 2017-01-18 00:00:00
URL: https://www.tmz.com/2017/01/18/lin-manuel-miranda-obama-commutes-oscar-lopez-rivera-sentence/


QueryResponse(matches=[ScoredVector(id='news-751', score=0.358225465, values=[], metadata={'article_content': "The media’s coverage of the 2016 race didn’t win the election for Donald Trump, but it didn’t help. Back in September, Gallup presented the findings of an important research project that, in retrospect, ought to have prompted a lot more soul searching among members of the press. What they did was, over an extended period of time, survey people and ask them what they were hearing about the two candidates. The answer is that with regard to Hillary Clinton, they heard a lot about email. With regard to Trump, they heard about nothing in particular. The way Frank Newport, Lisa Singh, Stuart Soroka, Michael Traugott, and Andrew Dugan put it for Gallup is that compared with the email-centric view of Clinton, “Americans' reports of what they have read, seen or heard about Donald Trump over this same period have been more varied and related to his campaign activities and statements.” J

In [20]:
def build_context_from_matches(matches) -> str:
    """
    Build context from top matched articles.
    """
    context_blocks = []

    for rank, match in enumerate(matches, start=1):
        metadata = match.metadata or {}

        block = f"""
ARTICLE {rank}
Similarity Score: {match.score}
Title: {metadata.get("title", "")}
Publication: {metadata.get("publication", "")}
Date: {metadata.get("date", "")}
URL: {metadata.get("url", "")}
Content:
{metadata.get("article_content", "")}
"""
        context_blocks.append(block.strip())

    return "\n\n---\n\n".join(context_blocks)


def ask_news_bot(query: str, top_k: int = 3) -> str:
    """
    Full two-stage semantic retrieval + GenAI answer generation.

    Stage 1: Search title vectors in Pinecone.
    Stage 2: Pull article content from metadata.
    Stage 3: Ask GPT to answer using retrieved context.
    """
    search_results = search_news_titles(query, top_k=top_k)

    if not search_results.matches:
        return "No relevant articles found."

    context = build_context_from_matches(search_results.matches)

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a careful news research assistant.
Answer the user's question using only the provided context.
If the context is not enough, say that the available articles do not provide
enough information.
Mention article titles or sources when useful.
"""
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{query}
"""
            }
        ]
    )

    return response.choices[0].message.content

In [21]:
answer = ask_news_bot("What happened with Obama?", top_k=3)
print(answer)

The available articles do not provide enough information to fully answer what happened with Obama. However, they do mention that President Obama criticized Donald Trump's policies in a speech, particularly focusing on national security and trade issues. Additionally, Obama commuted the sentence of Oscar Lopez Rivera, a controversial decision that led to criticism from some, including members of the NYPD.
